# Does moving to the city change how much someone posts? (Analysis 9)

Fits a distribution family to daily message counts (the `04.2`/`04.4`
technique), per mover, before vs. after their own move to the city —
rather than the categorical-share approach Analysis 8 used for hour-of-day.

**Proposition (Stage 1):** after moving to the city, a person's own daily
message rate in the group chat decreases — more things to do, a new social
life competing for attention. Same predicted direction for all 4 movers.

**Guarded against (the boring result):** Analysis 1 already found this
chat's overall volume declining since lockdown ended — so an individual
mover's before/after dip could just be riding that general chat-wide
decline, not something specific to them moving. Every mover's own ratio is
compared against the *rest of the group's* ratio over the identical
calendar window, not read in isolation.

Author aliases only — no real names, same anonymisation convention as every
other analysis in this project.


In [ ]:
import numpy as np
import pandas as pd

from goad_toolkit.analytics import DistributionFitter, FitResult
from goad_toolkit.distributions import DistributionRegistry
from goad_toolkit.visualizer import GroupedBarPlot, HorizontalLine, PlotSettings

from wa_analyzer.data import load_own_chat

own = load_own_chat()
own["timestamp"] = pd.to_datetime(own["timestamp"], utc=True).dt.tz_localize(None)
own["date"] = own["timestamp"].dt.date

EXPORT_START = own["timestamp"].min().date()
EXPORT_END = own["timestamp"].max().date()
print(EXPORT_START, "->", EXPORT_END)


## Movers and their move dates

Move dates are month-precision, from memory — day-of-month fixed to the 1st
as an approximation, same "static, approximate lookup" precedent as the
lockdown/election tables elsewhere in this project.


In [ ]:
MOVERS = {
    "striking-rail": "2021-08-01",
    "animated-elk": "2022-09-01",
    "effervescent-penguin": "2023-07-01",
    "humorous-stingray": "2023-09-01",
}

fitter = DistributionFitter(DistributionRegistry(), seed=42)


def daily_counts(frame: pd.DataFrame, start, end) -> np.ndarray:
    """One count per calendar day in [start, end), zero-filled -- the true rate, not just posting days."""
    window = frame[(frame["date"] >= start) & (frame["date"] < end)]
    counts = window.groupby("date").size()
    full_range = pd.date_range(start, end, freq="D", inclusive="left").date
    return counts.reindex(full_range, fill_value=0).to_numpy()


def fitted_mean(counts: np.ndarray) -> tuple[str, float]:
    fits = fitter.fit(counts, discrete=True)
    best = next(f for f in fits if isinstance(f, FitResult) and f.best_likelihood)
    return best.distribution, float(best.frozen_dist.mean())


## Fit each mover's own rate, and the rest-of-group's rate, before vs. after

Both sides get exactly the same calendar windows, so the group's own trend
over that same stretch is the control the Stage 1 boring-result check
demanded.


In [ ]:
rows = []
for alias, move_date in MOVERS.items():
    move_date = pd.Timestamp(move_date).date()
    person = own[own["author"] == alias]
    rest = own[own["author"] != alias]

    p_before = daily_counts(person, EXPORT_START, move_date)
    p_after = daily_counts(person, move_date, EXPORT_END)
    g_before = daily_counts(rest, EXPORT_START, move_date)
    g_after = daily_counts(rest, move_date, EXPORT_END)

    p_before_fam, p_before_mean = fitted_mean(p_before)
    p_after_fam, p_after_mean = fitted_mean(p_after)
    g_before_fam, g_before_mean = fitted_mean(g_before)
    g_after_fam, g_after_mean = fitted_mean(g_after)

    person_ratio = p_after_mean / p_before_mean
    group_ratio = g_after_mean / g_before_mean

    rows.append({
        "mover": alias,
        "n_before_days": len(p_before),
        "n_after_days": len(p_after),
        "person_before_rate": round(p_before_mean, 2),
        "person_after_rate": round(p_after_mean, 2),
        "person_ratio": round(person_ratio, 2),
        "group_before_rate": round(g_before_mean, 2),
        "group_after_rate": round(g_after_mean, 2),
        "group_ratio": round(group_ratio, 2),
        "relative_ratio": round(person_ratio / group_ratio, 2),
    })

summary = pd.DataFrame(rows)
summary


**Reading `relative_ratio`:** person's own after/before ratio divided by the
group's own after/before ratio over the same window. **1.0 means the person's
change is no different from the group's general trend over that time** —
below 1 means they slowed down *more* than the group did; above 1 means
*less*, or they picked up while the group declined.


## The single comparison this chart exists to make

Not "did their raw rate drop" (Analysis 1 already showed the whole chat's
rate drops over time) but **did it drop by more than the group's own
drop, over the identical stretch.**


In [ ]:
plot_data = summary.melt(
    id_vars="mover", value_vars=["person_ratio", "group_ratio"],
    var_name="series", value_name="ratio",
)
plot_data["series"] = plot_data["series"].map({
    "person_ratio": "this person", "group_ratio": "rest of group",
})

settings = PlotSettings(
    figsize=(9, 5),
    title="After-move rate \u00f7 before-move rate: this person vs. the rest of the group",
    xlabel="mover",
    ylabel="after-rate / before-rate",
    legend_title="",
)
bars = GroupedBarPlot(settings)
fig, ax = bars.plot(
    data=plot_data, x="mover", y="ratio", hue="series",
    palette={"this person": "#C44E52", "rest of group": "#4C72B0"},
)
lines_helper = HorizontalLine(settings)
lines_helper.ax = ax
lines_helper.fig = fig
lines_helper.build(y=1.0, color="#333333", linestyle="--", linewidth=1)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig("city-move-rate-ratio.png", dpi=150)


## Next: Stage 5/6

Look at the table and chart above the way the goad checklist's critique and
verification stages ask, before the next chat message answers those
questions — this is the honest read, not a description of what the code
above was trying to do.
